# Quantum Drude oscillators

Regenerate the Hamiltonian and use native Python quantum functions to perform VQE on two and three coupled oscillators. All eleven runs begin with angles equal to one and allow 500 optimization iterations. Adam trains the identical gate sequence using a differentiable FlagQuantum Circuit and a full statevector expectation. Each final binding is synthesized and run as a native qfunc, and its full state is compared with the training circuit. This workflow uses the source step budget and a local learning rate of 0.04. The reported finite-basis ground energy comes from an independent diagonalization.

Run in a Python environment with FlagQuantum, NumPy, Matplotlib and Jupyter installed. Artifacts are written below the Notebook working directory.


## Local setup

In [ ]:
from importlib import import_module
from pathlib import Path


In [ ]:
import numpy as np

from examples.synthesis.applications.chemistry.quantum_drude_oscillator.overview.main import (
    DIMENSIONS,
    show_run,
)
from examples.synthesis.applications.chemistry.quantum_drude_oscillator.overview.model import (
    POLARIZABILITY,
    finite_ground_energies,
    harmonic_ground_energy,
    ladder_projector,
    oscillator_hamiltonian,
    pairwise_interaction,
    position_operator,
)
from examples.synthesis.applications.chemistry.quantum_drude_oscillator.overview.plots import (
    save_energy_curves,
    save_trace,
)
from examples.synthesis.applications.chemistry.quantum_drude_oscillator.overview.quantum import (
    build_program,
)
from examples.synthesis.applications.chemistry.quantum_drude_oscillator.overview.reference import (
    analytical_position_product,
    transition_matrix,
)
from examples.synthesis.applications.chemistry.quantum_drude_oscillator.overview.workflow import (
    optimize_distance,
    sweep,
)

output = Path("artifacts/development/drude-oscillator")


def show_image(path: Path) -> None:
    display = import_module("IPython.display")
    display.display(display.Image(filename=str(path)))


## Occupation basis

In [ ]:
occupation = 2
print(f"Occupation {occupation} uses bits {occupation:02b}")


In [ ]:
next_occupation = occupation + 1
print(f"Raise {occupation:02b} to {next_occupation:02b}")


In [ ]:
raising = ladder_projector(occupation, 4, raising=True)
lowering = ladder_projector(occupation, 4, raising=False)


In [ ]:
assert raising[next_occupation, occupation] == 1
assert np.array_equal(raising.T, lowering)
print(raising)


In [ ]:
position = position_operator(4)
print(position)


In [ ]:
two_level = position_operator(2)
np.testing.assert_allclose(
    np.kron(two_level, two_level), analytical_position_product(1)
)


In [ ]:
np.testing.assert_allclose(
    np.kron(position, position), analytical_position_product(2), atol=1e-14
)


## Hamiltonian and four-qubit variational test

In [ ]:
bare_levels = 2 * np.arange(4) + 1
print(bare_levels)


In [ ]:
test_matrix = oscillator_hamiltonian(2, 4, 3.0)
np.testing.assert_allclose(test_matrix, transition_matrix(2, 4, 3.0), atol=1e-14)


In [ ]:
initial_two = (1.0,) * 12


In [ ]:
program_two = build_program(2, initial_two)
print(program_two.report["resources"])


In [ ]:
run = optimize_distance(2, 3.0)


In [ ]:
show_run(run)
show_image(save_trace(run, output / "vqe-trace.png"))


## Two oscillators

In [ ]:
sample_two = np.linspace(3.3, 5.5, 5)
print(sample_two)


In [ ]:
two = sweep(2)
for run in two:
    show_run(run)


In [ ]:
distances = np.arange(3.1, 6.1, 0.2)
harmonic = (
    ("Harmonic solution", harmonic_ground_energy(-4 * POLARIZABILITY / distances**3)),
)


In [ ]:
show_image(
    save_energy_curves(
        distances,
        harmonic,
        two,
        xlabel="Distance (Å)",
        reference_energy=2.0,
        title="Two-oscillator London dispersion",
        path=output / "two-oscillators.png",
    )
)


## Finite-basis truncation

In [ ]:
truncation = tuple(
    (f"d = {dimension}", finite_ground_energies(distances, dimension))
    for dimension in DIMENSIONS
)


In [ ]:
show_image(
    save_energy_curves(
        distances,
        (*truncation, *harmonic),
        (),
        xlabel="Distance (Å)",
        reference_energy=2.0,
        title="Finite-basis truncation by distance",
        path=output / "finite-distance.png",
    )
)


In [ ]:
coupling = -np.linspace(0.01, 2.0, 15)
coupling_distances = (-4 * POLARIZABILITY / coupling) ** (1 / 3)
coupling_curves = tuple(
    (f"d = {dimension}", finite_ground_energies(coupling_distances, dimension))
    for dimension in DIMENSIONS
)


In [ ]:
show_image(
    save_energy_curves(
        -coupling,
        (*coupling_curves, ("Harmonic solution", harmonic_ground_energy(coupling))),
        (),
        xlabel="Negative coupling",
        reference_energy=0.0,
        title="Finite-basis truncation by coupling",
        path=output / "finite-coupling.png",
    )
)


## Three oscillators

In [ ]:
pair_curve = 3 + pairwise_interaction(distances)
three_body_curve = finite_ground_energies(distances, 4, 3)


In [ ]:
program_three = build_program(3, (1.0,) * 40)
print(program_three.report["resources"])


In [ ]:
sample_three = np.linspace(3.1, 5.5, 5)
print(sample_three)


In [ ]:
three = sweep(3)
for run in three:
    show_run(run)


In [ ]:
show_image(
    save_energy_curves(
        distances,
        (
            ("Pair interactions", pair_curve),
            ("Finite three-body diagonalization", three_body_curve),
        ),
        three,
        xlabel="Distance (Å)",
        reference_energy=3.0,
        title="Three-oscillator London dispersion",
        path=output / "three-oscillators.png",
    )
)
